# Selected Models Against Selected Benchmarks

This notebook organizes the study models by family and the registered benchmarks by task family. Select one model list and one benchmark list, then run their Cartesian product with `run_benchmark_matrix`.

API models require their provider credentials. Local models require sufficient storage, RAM, and usually CUDA. Failures are recorded so one unavailable model does not stop the remaining runs.

## 1. Repository and run configuration

In [ ]:
from pathlib import Path
import gc
import inspect
import json
import os
import subprocess
import sys
import traceback

import pandas as pd
import torch

REPO_URL = "https://github.com/samhatescoding/transformers.git"
REPO_BRANCH = "main"

current_dir = Path.cwd()
if (current_dir / "models").is_dir():
    REPO_ROOT = current_dir
else:
    clone_parent = Path("/workspace") if Path("/workspace").is_dir() else current_dir
    REPO_ROOT = clone_parent / "transformers"
    if REPO_ROOT.exists():
        if not (REPO_ROOT / "models").is_dir():
            raise RuntimeError(f"Clone target exists but is not the repository: {REPO_ROOT}")
    else:
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                REPO_BRANCH,
                "--single-branch",
                REPO_URL,
                str(REPO_ROOT),
            ],
            check=True,
        )
REPO_ROOT = REPO_ROOT.resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NUM_SAMPLES = 2
MAX_NEW_TOKENS = 16
OUTPUT_DIR = REPO_ROOT / "results" / "selected_model_benchmark_matrix"
SUMMARY_PATH = OUTPUT_DIR / "summary.json"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Results: {OUTPUT_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")

Set provider keys in the environment before running. Common variables include `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`, `DASHSCOPE_API_KEY`, and any custom provider base URL/key required by a wrapper.

In [ ]:
for variable in (
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "GOOGLE_API_KEY",
    "DASHSCOPE_API_KEY",
    "HF_TOKEN",
):
    print(f"{variable}: {'set' if os.getenv(variable) else 'not set'}")

## 2. List the selected models by family

Each entry contains the result name and its model wrapper class. Edit the short family lists directly when the study selection changes.

In [ ]:
from models import (
    ClaudeOpus46,
    ClaudeOpus47,
    ClaudeOpus48,
    ClaudeSonnet45,
    ClaudeSonnet46,
    DotsOCR,
    DotsVLM1,
    Gemini31ProPreview,
    Gemini35Flash,
    Gemma4_26BA4B,
    Gemma4_31B,
    GLM41VThinking,
    GLM45VThinking,
    GPT54,
    GPT55,
    KimiK25Thinking,
    KimiK26,
    MiMoV25,
    MiMoV25Pro,
    MistralMedium35,
    MistralSmall4,
    O3,
    O4Mini,
    Qwen35_397BA17B,
    Qwen36_27B,
    Qwen36Plus,
    Qwen37PlusPreview,
    Qwen3VL235BA22B,
    SkyworkR1V2_38B,
    SkyworkR1V3_38B,
)

OPENAI_MODELS = [
    ("gpt-5.5", GPT55),
    ("gpt-5.4", GPT54),
    ("o3", O3),
    ("o4-mini", O4Mini),
]

CLAUDE_OPUS_MODELS = [
    ("claude-opus-4-8", ClaudeOpus48),
    ("claude-opus-4-7", ClaudeOpus47),
    ("claude-opus-4-6", ClaudeOpus46),
]

CLAUDE_SONNET_MODELS = [
    ("claude-sonnet-4-6", ClaudeSonnet46),
    ("claude-sonnet-4-5", ClaudeSonnet45),
]

GEMINI_MODELS = [
    ("gemini-3.1-pro-preview", Gemini31ProPreview),
    ("gemini-3.5-flash", Gemini35Flash),
]

QWEN_MODELS = [
    ("qwen3.7-plus-preview", Qwen37PlusPreview),
    ("qwen3.6-plus", Qwen36Plus),
    ("qwen3.6-27b", Qwen36_27B),
    ("qwen3.5-397b-a17b", Qwen35_397BA17B),
    ("qwen3-vl-235b-a22b", Qwen3VL235BA22B),
]

KIMI_MODELS = [
    ("kimi-k2.6", KimiK26),
    ("kimi-k2.5-thinking", KimiK25Thinking),
]

GEMMA_MODELS = [
    ("gemma-4-31b", Gemma4_31B),
    ("gemma-4-26b-a4b", Gemma4_26BA4B),
]

MIMO_MODELS = [
    ("mimo-v2.5", MiMoV25),
    ("mimo-v2.5-pro", MiMoV25Pro),
]

MISTRAL_MODELS = [
    ("mistral-medium-3.5", MistralMedium35),
    ("mistral-small-4", MistralSmall4),
]

DOTS_MODELS = [
    ("dots.vlm1", DotsVLM1),
    ("dots.ocr", DotsOCR),
]

GLM_MODELS = [
    ("glm-4.5v-thinking", GLM45VThinking),
    ("glm-4.1v-thinking", GLM41VThinking),
]

SKYWORK_MODELS = [
    ("skywork-r1v3-38b", SkyworkR1V3_38B),
    ("skywork-r1v2-38b", SkyworkR1V2_38B),
]

MODEL_FAMILIES = {
    "OpenAI": OPENAI_MODELS,
    "Claude Opus": CLAUDE_OPUS_MODELS,
    "Claude Sonnet": CLAUDE_SONNET_MODELS,
    "Gemini": GEMINI_MODELS,
    "Qwen": QWEN_MODELS,
    "Kimi": KIMI_MODELS,
    "Gemma": GEMMA_MODELS,
    "MiMo": MIMO_MODELS,
    "Mistral": MISTRAL_MODELS,
    "dots": DOTS_MODELS,
    "GLM": GLM_MODELS,
    "Skywork": SKYWORK_MODELS,
}

model_count = sum(len(family_models) for family_models in MODEL_FAMILIES.values())
print(f"Configured {model_count} models across {len(MODEL_FAMILIES)} families.")
{family: [name for name, _ in family_models] for family, family_models in MODEL_FAMILIES.items()}

## 3. List the benchmarks by task family

Each list contains benchmark classes. The task-family dictionary makes it easy to select a complete group or construct a custom list.

In [ ]:
from benchmarks import (
    BLIP3o60kBenchmark,
    CityscapesBenchmark,
    ConceptualCaptionsBenchmark,
    ConceptualCaptionsCaptionBenchmark,
    DFDCBenchmark,
    DiffusionDBBenchmark,
    DocVQABenchmark,
    FairFaceBenchmark,
    FashionMNISTBenchmark,
    Flickr30kBenchmark,
    Flickr30kEntitiesBenchmark,
    FlyingThings3DBenchmark,
    GQABenchmark,
    HDTFBenchmark,
    HQEditBenchmark,
    ImageNet1kBenchmark,
    ImgEditBenchmark,
    INaturalistBenchmark,
    InternVidBenchmark,
    KineticsBenchmark,
    LAION400MBenchmark,
    LAION5BBenchmark,
    LSUNBenchmark,
    LVISBenchmark,
    MagicBrushBenchmark,
    MSCOCOBenchmark,
    MSCOCOCaptionBenchmark,
    MVTecADBenchmark,
    OpenImagesV4Benchmark,
    OpenImagesV4DetectionBenchmark,
    OpenVid1MBenchmark,
    PickAPicBenchmark,
    PlacesBenchmark,
    ShareGPT4oImageBenchmark,
    TAD66KBenchmark,
    TextCapsBenchmark,
    UCF101Benchmark,
    VisualCoTBenchmark,
    VisualGenomeBenchmark,
    VQAv2Benchmark,
)

CLASSIFICATION_BENCHMARKS = [
    CityscapesBenchmark,
    FairFaceBenchmark,
    FashionMNISTBenchmark,
    ImageNet1kBenchmark,
    INaturalistBenchmark,
    LSUNBenchmark,
    MVTecADBenchmark,
    OpenImagesV4Benchmark,
    PlacesBenchmark,
]

DETECTION_BENCHMARKS = [
    Flickr30kEntitiesBenchmark,
    LVISBenchmark,
    MSCOCOBenchmark,
    OpenImagesV4DetectionBenchmark,
]

CAPTIONING_BENCHMARKS = [
    ConceptualCaptionsCaptionBenchmark,
    Flickr30kBenchmark,
    HDTFBenchmark,
    InternVidBenchmark,
    LAION400MBenchmark,
    LAION5BBenchmark,
    MSCOCOCaptionBenchmark,
    TextCapsBenchmark,
]

VISUAL_QA_BENCHMARKS = [
    DocVQABenchmark,
    GQABenchmark,
    VisualCoTBenchmark,
    VisualGenomeBenchmark,
    VQAv2Benchmark,
]

VIDEO_CLASSIFICATION_BENCHMARKS = [
    DFDCBenchmark,
    KineticsBenchmark,
    UCF101Benchmark,
]

IMAGE_MODIFICATION_VQA_BENCHMARKS = [
    HQEditBenchmark,
    ImgEditBenchmark,
    MagicBrushBenchmark,
]

IMAGE_JUDGMENT_BENCHMARKS = [
    PickAPicBenchmark,
    TAD66KBenchmark,
]

PROMPT_AND_MULTIPLE_CHOICE_BENCHMARKS = [
    BLIP3o60kBenchmark,
    ConceptualCaptionsBenchmark,
    DiffusionDBBenchmark,
    FlyingThings3DBenchmark,
    OpenVid1MBenchmark,
    ShareGPT4oImageBenchmark,
]

BENCHMARK_FAMILIES = {
    "Classification": CLASSIFICATION_BENCHMARKS,
    "Detection": DETECTION_BENCHMARKS,
    "Captioning": CAPTIONING_BENCHMARKS,
    "Visual QA": VISUAL_QA_BENCHMARKS,
    "Video classification": VIDEO_CLASSIFICATION_BENCHMARKS,
    "Image modification VQA": IMAGE_MODIFICATION_VQA_BENCHMARKS,
    "Image judgment": IMAGE_JUDGMENT_BENCHMARKS,
    "Prompt and multiple choice": PROMPT_AND_MULTIPLE_CHOICE_BENCHMARKS,
}

benchmark_count = sum(len(items) for items in BENCHMARK_FAMILIES.values())
print(f"Configured {benchmark_count} benchmarks across {len(BENCHMARK_FAMILIES)} task families.")
{family: [cls.benchmark_name for cls in classes] for family, classes in BENCHMARK_FAMILIES.items()}

## 4. Select one model list and one benchmark list

The following is placeholder configuration. Change the two selected lists to any entries in `MODEL_FAMILIES` and `BENCHMARK_FAMILIES`.

In [ ]:
from runners.benchmark_run import BenchmarkRun
from runners.execution import run_benchmark_matrix
from runners.model_run import ModelRun

def model_kwargs(model_class):
    parameters = inspect.signature(model_class).parameters
    kwargs = {"max_new_tokens": MAX_NEW_TOKENS}
    if "stream" in parameters:
        kwargs["stream"] = False
    return kwargs

# Placeholder selection: replace these with any family lists defined above.
SELECTED_MODELS = GEMINI_MODELS
SELECTED_BENCHMARKS = VISUAL_QA_BENCHMARKS

model_runs = [
    ModelRun.from_factory(
        model_name,
        model_class,
        **model_kwargs(model_class),
    )
    for model_name, model_class in SELECTED_MODELS
]

benchmark_runs = [
    BenchmarkRun(
        benchmark=benchmark_class(streaming=True),
        num_samples=NUM_SAMPLES,
    )
    for benchmark_class in SELECTED_BENCHMARKS
]

print(f"Models: {[run.name for run in model_runs]}")
print(f"Benchmarks: {[run.benchmark.name for run in benchmark_runs]}")
print(f"Matrix pairs: {len(model_runs) * len(benchmark_runs)}")

summary = run_benchmark_matrix(
    models=model_runs,
    benchmark_runs=benchmark_runs,
    output_dir=OUTPUT_DIR,
)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary_df = pd.DataFrame(summary)
summary_df

## 5. Inspect the completed matrix

In [ ]:
print(f"Completed pairs: {len(summary_df)}")
summary_df[["model", "benchmark", "num_samples", "results_path"]]